#### Setup

import boto3
from pathlib import Path
import sys

sys.path.append("../src")

from text_cleaning import clean_text

#### S3 paths

In [5]:
BUCKET_NAME = "multiomic-vae-literature-rag-123223178042-eu-north-1-an"

RAW_TEXT_PREFIX = "papers/text/"
CLEAN_TEXT_PREFIX = "papers/clean_text/"

s3 = boto3.client("s3")

#### List all extracted text files

In [6]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=RAW_TEXT_PREFIX
)

text_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].lower().endswith(".txt")
]

print(f"Found {len(text_files)} text files.")

Found 34 text files.


#### Clean all files and save to S3

In [7]:
for key in text_files:
    filename = Path(key).name
    output_key = CLEAN_TEXT_PREFIX + filename

    obj = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key
    )

    raw_text = obj["Body"].read().decode("utf-8", errors="ignore")
    cleaned_text = clean_text(raw_text)

    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=output_key,
        Body=cleaned_text.encode("utf-8"),
        ContentType="text/plain"
    )

    print(f"Saved: {output_key}")

Saved: papers/clean_text/BindVAE.txt
Saved: papers/clean_text/CASTLE.txt
Saved: papers/clean_text/CAVACHON.txt
Saved: papers/clean_text/Chromatin_GeneRegulation_Review.txt
Saved: papers/clean_text/Cicero.txt
Saved: papers/clean_text/GNODEVAE.txt
Saved: papers/clean_text/GenKI.txt
Saved: papers/clean_text/JAMIE.txt
Saved: papers/clean_text/LiVAE.txt
Saved: papers/clean_text/MultiVI.txt
Saved: papers/clean_text/Papers___MDPI_GENES___GAGAM_v1_2__an_improvement_on_peak_labeling_and_Genomic_Annotated_Gene_Activity_Matrix_construction.txt
Saved: papers/clean_text/SCA.txt
Saved: papers/clean_text/UnionCom.txt
Saved: papers/clean_text/VAE_BatchCorrection_scRNAseq_Benchmark.txt
Saved: papers/clean_text/biVI.txt
Saved: papers/clean_text/cobolt.txt
Saved: papers/clean_text/factVAE.txt
Saved: papers/clean_text/hybridVI.txt
Saved: papers/clean_text/pair.txt
Saved: papers/clean_text/peakVI.txt
Saved: papers/clean_text/phd-ConvNet-VAE.txt
Saved: papers/clean_text/phd-scPair.txt
Saved: papers/clean_te

#### Verify cleaned files count

In [8]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=CLEAN_TEXT_PREFIX
)

clean_files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].lower().endswith(".txt")
]

print(f"Found {len(clean_files)} cleaned text files.")

Found 34 cleaned text files.
